In [24]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data" / "raw"

adata = sc.read_h5ad(DATA_DIR / "clustered_data.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 28557 × 13727
    obs: 'sample', 'donor', 'condition', 'cell_type', 'multiplets', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'leiden'
    var: 'gene_symbol', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std'
    uns: 'cell_type_colors', 'condition_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'


In [25]:
# subset to only the highly variable genes
adata_hvg = adata[:, adata.var['highly_variable']].copy()
print(adata_hvg.shape)

(28557, 2000)


In [26]:
# subset to monocytes

mono = adata_hvg[adata_hvg.obs['cell_type'] == 'CD14+ Monocytes'].copy()
print(mono.shape)
print(mono.obs['condition'].value_counts())

(6325, 2000)
condition
ctrl    3335
stim    2990
Name: count, dtype: int64


In [27]:
X = np.array(mono.X)  # Convert sparse matrix to dense array
y = (mono.obs['condition'] == "stim").astype(int).values  # Extract the target variable
print(X.shape)
print(y.sum(), "stim cells")
print((y == 0).sum(), "ctrl cells")

(6325, 2000)
2990 stim cells
3335 ctrl cells


In [28]:
donors = mono.obs['donor'].unique()
print("Donors:", donors)

Donors: [1016. 1256. 1015.  107. 1244.  101. 1488. 1039.]


In [29]:
print(mono.obs['donor'].value_counts())

donor
1015.0    1675
1256.0     883
1016.0     853
1244.0     850
1488.0     748
101.0      539
107.0      462
1039.0     315
Name: count, dtype: int64


In [30]:
# donor-aware train-test split
train_donors = [1015., 1256.]  # Use the first two donors for training
test_donors = [d for d in donors if d not in train_donors]  # Use the next six donors for testing

train_mask = mono.obs['donor'].isin(train_donors)
test_mask = mono.obs['donor'].isin(test_donors)

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (2558, 2000), Test: (3767, 2000)


In [31]:
# train classifier
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_train, y_train)
print("Training complete!")

Training complete!


In [32]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["ctrl", "stim"]))

              precision    recall  f1-score   support

        ctrl       1.00      1.00      1.00      1959
        stim       1.00      1.00      1.00      1808

    accuracy                           1.00      3767
   macro avg       1.00      1.00      1.00      3767
weighted avg       1.00      1.00      1.00      3767



In [34]:
print(mono.obs.groupby(["donor", "condition"]).size().unstack())

condition  ctrl  stim
donor                
101.0       246   293
107.0       254   208
1015.0      907   768
1016.0      461   392
1039.0      134   181
1244.0      500   350
1256.0      469   414
1488.0      364   384


/var/folders/nr/2djqm8nd3bzbs_nt753qc5qc0000gn/T/ipykernel_60583/3139820769.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(mono.obs.groupby(["donor", "condition"]).size().unstack())


In [35]:
results = {}

cell_types = adata_hvg.obs['cell_type'].unique()

for ct in cell_types:
    subset = adata_hvg[adata_hvg.obs['cell_type'] == ct].copy()
    X = np.array(subset.X)
    y = (subset.obs['condition'] == "stim").astype(int).values

    donors = subset.obs['donor'].unique()
    donor_sizes = subset.obs['donor'].value_counts()

    # use 2 mid-sized donors for testing, and the rest for training
    sorted_donors = donor_sizes.sort_values().index
    test_donors = sorted_donors[len(sorted_donors)//4 : len(sorted_donors)//4 + 2]
    train_donors = [d for d in donors if d not in test_donors]

    train_mask = subset.obs['donor'].isin(train_donors)
    test_mask = subset.obs['donor'].isin(test_donors)

    X_train = X[train_mask]
    y_train = y[train_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]

    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    report = classification_report(y_test, y_pred, target_names=["ctrl", "stim"], output_dict=True)
    results[ct] = {
        "clf": clf,
        "X_test": X_test,
        "y_test": y_test,
        "f1": report["macro avg"]["f1-score"],
        "gene_names": adata_hvg.var_names.tolist()
    }

    print(f"Cell type: {ct}, F1-score: {report['macro avg']['f1-score']:.3f}")

Cell type: CD4 T cells, F1-score: 0.963
Cell type: CD14+ Monocytes, F1-score: 0.998
Cell type: Dendritic cells, F1-score: 0.959
Cell type: NK cells, F1-score: 0.970
Cell type: CD8 T cells, F1-score: 0.980
Cell type: B cells, F1-score: 0.988
Cell type: Megakaryocytes, F1-score: 0.717
Cell type: FCGR3A+ Monocytes, F1-score: 0.997
